# 40 · Residual Independence and Completeness Test

This notebook pivots from “can residual corrections improve the model?” to a stricter question:

\[
\text{Does the residual contain any independent, reusable signal beyond the unified core?}
\]

Notebooks 35–39 found that:
- residual corrections did not materially improve the unified boundary
- weighted residuals saturated or collapsed
- learned-\(\lambda\) and phase/error gating did not unlock stable gains

That suggests a new diagnostic path:
1. test whether the residual is predictable from the unified feature space
2. test whether the residual tracks unified uncertainty/error
3. test whether the residual transfers across conditions
4. remove the part of the residual explained by the unified feature span
5. test whether the orthogonalized residual still helps

## Main objects

For each task we define:
- core score: \(s_{\mathrm{core}}(x)\)
- specialist score: \(s_{\mathrm{spec}}(x)\)
- residual: \(r(x) = s_{\mathrm{spec}}(x) - s_{\mathrm{core}}(x)\)

## Main outputs

- residual predictability \(R^2\) across scales
- residual vs core-uncertainty correlation summaries
- cross-condition transfer heatmaps
- orthogonal residual gain heatmaps
- residual-only classifier comparison
- completeness summary table

In [ ]:
import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt

mp.mp.dps = 50
rng = np.random.default_rng(9423)

## Base data

In [ ]:
N = 2200
zeros = [mp.zetazero(n) for n in range(1, N + 1)]
t = np.array([float(mp.im(z)) for z in zeros])
zeta_gaps_full = np.diff(t)

poisson_gaps_full = rng.exponential(scale=1.0, size=len(zeta_gaps_full))

def sample_gue_bulk_spacings(matrix_size=140, n_matrices=70, bulk_fraction=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    all_gaps = []
    for _ in range(n_matrices):
        real = rng.normal(size=(matrix_size, matrix_size))
        imag = rng.normal(size=(matrix_size, matrix_size))
        A = real + 1j * imag
        H = (A + A.conj().T) / 2.0
        eigvals = np.linalg.eigvalsh(H)
        eigvals = np.sort(eigvals)
        n = len(eigvals)
        keep = int(n * bulk_fraction)
        start = (n - keep) // 2
        stop = start + keep
        bulk_vals = eigvals[start:stop]
        all_gaps.extend(np.diff(bulk_vals).tolist())
    return np.array(all_gaps)

gue_gaps_full = sample_gue_bulk_spacings(matrix_size=140, n_matrices=70, bulk_fraction=0.5, rng=rng)

len(zeta_gaps_full), len(poisson_gaps_full), len(gue_gaps_full)

## Window statistics and feature builder

In [ ]:
def extract_windows(gaps, k=5):
    return np.array([gaps[i:i+k] for i in range(len(gaps) - k + 1)])

def normalize_window(window):
    w = np.array(window, dtype=float)
    return w / w.mean()

def unevenness(window):
    return float(np.max(window) - np.min(window))

def reversal_symmetry_score(window):
    return float(np.mean(np.abs(window - window[::-1])))

def window_entropy(window):
    eps = 1e-12
    p = window / np.sum(window)
    return float(-np.sum(p * np.log(p + eps)))

def ratio_mean(window):
    g1 = window[:-1]
    g2 = window[1:]
    r = np.minimum(g1, g2) / np.maximum(g1, g2)
    return float(np.mean(r))

def ratio_std(window):
    g1 = window[:-1]
    g2 = window[1:]
    r = np.minimum(g1, g2) / np.maximum(g1, g2)
    return float(np.std(r))

def build_windows_and_features(gaps, feature_family="minimal", k=5):
    windows = extract_windows(gaps, k=k)
    windows_n = np.array([normalize_window(w) for w in windows])

    stats = {
        "entropy": np.array([window_entropy(w) for w in windows_n]),
        "unevenness": np.array([unevenness(w) for w in windows_n]),
        "ratio_mean": np.array([ratio_mean(w) for w in windows_n]),
        "symmetry": np.array([reversal_symmetry_score(w) for w in windows_n]),
    }

    if feature_family == "minimal":
        X = np.array([
            [stats["entropy"][i], stats["unevenness"][i], stats["ratio_mean"][i]]
            for i in range(len(windows_n))
        ], dtype=float)
    elif feature_family == "full":
        X = np.array([
            [stats["entropy"][i], stats["unevenness"][i], stats["symmetry"][i], stats["ratio_mean"][i], ratio_std(windows_n[i])]
            for i in range(len(windows_n))
        ], dtype=float)
    elif feature_family == "raw_window":
        X = windows_n.copy()
    else:
        raise ValueError(f"Unknown feature family: {feature_family}")

    return windows_n, stats, X

def apply_condition_mask(stats, condition_name):
    if condition_name == "low_entropy":
        thr = np.median(stats["entropy"])
        return stats["entropy"] <= thr
    if condition_name == "high_entropy":
        thr = np.median(stats["entropy"])
        return stats["entropy"] > thr
    if condition_name == "low_unevenness":
        thr = np.median(stats["unevenness"])
        return stats["unevenness"] <= thr
    if condition_name == "high_unevenness":
        thr = np.median(stats["unevenness"])
        return stats["unevenness"] > thr
    raise ValueError(condition_name)

def make_contextual_features(X):
    prev_X = np.vstack([X[0], X[:-1]])
    next_X = np.vstack([X[1:], X[-1]])
    delta_prev = X - prev_X
    delta_next = next_X - X
    curv = next_X - 2 * X + prev_X
    return np.hstack([X, delta_prev, delta_next, curv])

def standardize_train_test(X_train, X_test):
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std[std == 0] = 1.0
    return (X_train - mean) / std, (X_test - mean) / std, mean, std

## Core helpers

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -40, 40)))

def fit_logistic_regression(X, y, lr=0.1, n_steps=2500, reg=1e-4):
    Xb = np.column_stack([np.ones(len(X)), X])
    w = np.zeros(Xb.shape[1])
    for _ in range(n_steps):
        p = sigmoid(Xb @ w)
        grad = Xb.T @ (p - y) / len(X)
        grad[1:] += reg * w[1:]
        w -= lr * grad
    return w

def decision_function_logistic(X, w):
    Xb = np.column_stack([np.ones(len(X)), X])
    return Xb @ w

def predict_proba_logistic(X, w):
    return sigmoid(decision_function_logistic(X, w))

def fit_linear_regression(X, y, reg=1e-4):
    Xb = np.column_stack([np.ones(len(X)), X])
    I = np.eye(Xb.shape[1])
    I[0, 0] = 0.0
    beta = np.linalg.solve(Xb.T @ Xb + reg * I, Xb.T @ y)
    return beta

def predict_linear_regression(X, beta):
    Xb = np.column_stack([np.ones(len(X)), X])
    return Xb @ beta

def r2_score(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    if ss_tot <= 1e-12:
        return 0.0
    return float(1.0 - ss_res / ss_tot)

def correlation(x, y):
    x = np.asarray(x)
    y = np.asarray(y)
    if np.std(x) <= 1e-12 or np.std(y) <= 1e-12:
        return 0.0
    return float(np.corrcoef(x, y)[0,1])

def overlap_coefficient_from_hist(a, b, bins=30):
    lo = min(a.min(), b.min())
    hi = max(a.max(), b.max())
    if hi <= lo:
        return 1.0
    hist_a, edges = np.histogram(a, bins=bins, range=(lo, hi), density=True)
    hist_b, _ = np.histogram(b, bins=bins, range=(lo, hi), density=True)
    dx = edges[1] - edges[0]
    return float(np.sum(np.minimum(hist_a, hist_b)) * dx)

def evaluate_scores(y_true, scores, probs):
    preds = (probs >= 0.5).astype(int)
    acc = float(np.mean(preds == y_true))
    s0 = scores[y_true == 0]
    s1 = scores[y_true == 1]
    overlap = overlap_coefficient_from_hist(s0, s1, bins=30)
    return {"accuracy": acc, "overlap": overlap}

## Boundary model and residual extractor

In [ ]:
def choose_prototypes(X, n_proto=20):
    idx = np.linspace(0, len(X)-1, n_proto).astype(int)
    return X[idx]

def estimate_rbf_gamma(X):
    D = np.linalg.norm(X[:, None, :] - X[None, :, :], axis=2)
    pos = D[D > 0]
    med = np.median(pos) if len(pos) else 1.0
    if med <= 0:
        med = 1.0
    return 1.0 / (2.0 * med * med)

def rbf_features(X, prototypes, gamma):
    D2 = np.sum((X[:, None, :] - prototypes[None, :, :]) ** 2, axis=2)
    return np.exp(-gamma * D2)

def fit_rbf_boundary(X0_train, X1_train):
    m = min(len(X0_train), len(X1_train))
    X0_train = X0_train[:m]
    X1_train = X1_train[:m]
    X_train = np.vstack([X0_train, X1_train])
    y_train = np.array([0] * len(X0_train) + [1] * len(X1_train))

    Xtr_std, _, mean, std = standardize_train_test(X_train, X_train)
    prototypes = choose_prototypes(Xtr_std, n_proto=min(20, len(Xtr_std)))
    gamma = estimate_rbf_gamma(Xtr_std)
    R_train = rbf_features(Xtr_std, prototypes, gamma)
    w = fit_logistic_regression(R_train, y_train, lr=0.15, n_steps=3500, reg=1e-4)
    return {"mean": mean, "std": std, "prototypes": prototypes, "gamma": gamma, "w": w}

def boundary_scores(model, X0_test, X1_test):
    m = min(len(X0_test), len(X1_test))
    X0_test = X0_test[:m]
    X1_test = X1_test[:m]
    X_test = np.vstack([X0_test, X1_test])
    y_test = np.array([0] * len(X0_test) + [1] * len(X1_test))
    Xte_std = (X_test - model["mean"]) / model["std"]
    R_test = rbf_features(Xte_std, model["prototypes"], model["gamma"])
    scores = decision_function_logistic(R_test, model["w"])
    probs = predict_proba_logistic(R_test, model["w"])
    return y_test, scores, probs, X_test

def extract_core_and_residual(X0_train, X1_train, X0_test, X1_test):
    core_model = fit_rbf_boundary(X0_train, X1_train)
    spec_model = fit_rbf_boundary(X0_train, X1_train)

    y_true, s_core, p_core, X_test = boundary_scores(core_model, X0_test, X1_test)
    _, s_spec, _, _ = boundary_scores(spec_model, X0_test, X1_test)

    residual = s_spec - s_core
    residual = residual / (np.std(residual) + 1e-6)

    core_margin = np.abs(s_core)
    core_uncertainty = p_core * (1.0 - p_core)
    core_error = ((p_core >= 0.5).astype(int) != y_true).astype(float)

    return {
        "y_true": y_true,
        "X_test": X_test,
        "s_core": s_core,
        "p_core": p_core,
        "residual": residual,
        "core_margin": core_margin,
        "core_uncertainty": core_uncertainty,
        "core_error": core_error,
    }

## Residual diagnostics

In [ ]:
def predict_residual_from_features(X_train, r_train, X_test, use_context=False):
    if use_context:
        Xtr = make_contextual_features(X_train)
        Xte = make_contextual_features(X_test)
    else:
        Xtr = X_train
        Xte = X_test

    Xtr_std, Xte_std, _, _ = standardize_train_test(Xtr, Xte)
    beta = fit_linear_regression(Xtr_std, r_train, reg=1e-3)
    r_pred = predict_linear_regression(Xte_std, beta)
    return r_pred

def orthogonal_residual(X_train, r_train, X_test, r_test, use_context=False):
    r_pred = predict_residual_from_features(X_train, r_train, X_test, use_context=use_context)
    r_orth = r_test - r_pred
    return r_pred, r_orth

def residual_only_classifier(r_train, y_train, r_test, y_test):
    Xtr = r_train.reshape(-1, 1)
    Xte = r_test.reshape(-1, 1)
    Xtr_std, Xte_std, _, _ = standardize_train_test(Xtr, Xte)
    w = fit_logistic_regression(Xtr_std, y_train, lr=0.1, n_steps=1500, reg=1e-4)
    scores = decision_function_logistic(Xte_std, w)
    probs = predict_proba_logistic(Xte_std, w)
    return evaluate_scores(y_test, scores, probs)

def evaluate_orthogonal_gain(s_core, y_true, r_orth, alpha_grid=None):
    if alpha_grid is None:
        alpha_grid = np.linspace(0.0, 1.5, 16)
    base_eval = evaluate_scores(y_true, s_core, sigmoid(s_core))
    best_gain = -np.inf
    best_alpha = 0.0
    best_eval = base_eval
    for a in alpha_grid:
        s_new = s_core + a * r_orth
        cur_eval = evaluate_scores(y_true, s_new, sigmoid(s_new))
        gain = cur_eval["overlap"] - base_eval["overlap"]
        if gain > best_gain:
            best_gain = gain
            best_alpha = float(a)
            best_eval = cur_eval
    return best_alpha, best_gain, best_eval

## Experiment grid

In [ ]:
window_sizes = [3, 5, 7]
feature_family = "minimal"
sample_size = 100
height_block = (0, 400)
alpha_grid = np.linspace(0.0, 1.5, 16)

systems = {
    "entropy": ("low_entropy", "high_entropy"),
    "unevenness": ("low_unevenness", "high_unevenness"),
}
tasks = {
    "zeta_vs_GUE": ("zeta", "gue"),
    "zeta_vs_Poisson": ("zeta", "poisson"),
}

## Base gap slices

In [ ]:
start, stop = height_block
base_gaps = {
    "zeta": zeta_gaps_full[start:stop],
    "poisson": poisson_gaps_full[start:stop],
    "gue": gue_gaps_full[:max(stop - start + 300, 900)],
}
{k: len(v) for k, v in base_gaps.items()}

## Build condition-specific feature sets

In [ ]:
def get_conditioned_features(gaps, condition_name, k=5, feature_family="minimal", n=100):
    _, stats, X = build_windows_and_features(gaps, feature_family=feature_family, k=k)
    mask = apply_condition_mask(stats, condition_name)
    Xc = X[mask]
    return Xc[:min(len(Xc), n)]

conditioned = {}
for k in window_sizes:
    conditioned[k] = {}
    for cond_lo, cond_hi in systems.values():
        for cond in [cond_lo, cond_hi]:
            for ensemble_name, gaps in base_gaps.items():
                conditioned[k][(ensemble_name, cond)] = get_conditioned_features(
                    gaps, cond, k=k, feature_family=feature_family, n=sample_size
                )

{k: {key: len(val) for key, val in conditioned[k].items()} for k in window_sizes}

## Main completeness sweep

In [ ]:
results = []

for system_name, (cond_lo, cond_hi) in systems.items():
    for task_name, (ens0, ens1) in tasks.items():
        for k in window_sizes:
            X0_low = conditioned[k][(ens0, cond_lo)]
            X1_low = conditioned[k][(ens1, cond_lo)]
            X0_high = conditioned[k][(ens0, cond_hi)]
            X1_high = conditioned[k][(ens1, cond_hi)]

            m = min(len(X0_low), len(X1_low), len(X0_high), len(X1_high), sample_size)
            if m < 40:
                continue

            X0_low = X0_low[:m]
            X1_low = X1_low[:m]
            X0_high = X0_high[:m]
            X1_high = X1_high[:m]

            train0 = np.vstack([X0_low, X0_high])
            train1 = np.vstack([X1_low, X1_high])

            for eval_label, X0_test, X1_test, X0_train, X1_train, X0_cross, X1_cross in [
                ("low", X0_low, X1_low, X0_low, X1_low, X0_high, X1_high),
                ("high", X0_high, X1_high, X0_high, X1_high, X0_low, X1_low),
                ("mixed", np.vstack([X0_low, X0_high]), np.vstack([X1_low, X1_high]), train0, train1, np.vstack([X0_low, X0_high]), np.vstack([X1_low, X1_high])),
            ]:
                data = extract_core_and_residual(X0_train, X1_train, X0_test, X1_test)

                X_test = data["X_test"]
                y_true = data["y_true"]
                s_core = data["s_core"]
                residual = data["residual"]

                r_pred_basic = predict_residual_from_features(X_test, residual, X_test, use_context=False)
                r_pred_ctx = predict_residual_from_features(X_test, residual, X_test, use_context=True)
                r2_basic = r2_score(residual, r_pred_basic)
                r2_ctx = r2_score(residual, r_pred_ctx)

                corr_unc = correlation(np.abs(residual), data["core_uncertainty"])
                corr_margin = correlation(np.abs(residual), 1.0 / (1.0 + data["core_margin"]))
                corr_err = correlation(np.abs(residual), data["core_error"])

                cross_data_train = extract_core_and_residual(X0_train, X1_train, X0_train, X1_train)
                cross_data_test = extract_core_and_residual(X0_train, X1_train, X0_cross, X1_cross)
                r_transfer_pred = predict_residual_from_features(
                    cross_data_train["X_test"], cross_data_train["residual"], cross_data_test["X_test"], use_context=True
                )
                transfer_r2 = r2_score(cross_data_test["residual"], r_transfer_pred)

                _, r_orth = orthogonal_residual(X_test, residual, X_test, residual, use_context=True)
                best_alpha, orth_gain, orth_eval = evaluate_orthogonal_gain(s_core, y_true, r_orth, alpha_grid=alpha_grid)

                resid_only = residual_only_classifier(residual, y_true, residual, y_true)

                if r2_ctx > 0.65 and orth_gain < 0.01 and resid_only["overlap"] < 0.58:
                    verdict = "redundant"
                elif orth_gain > 0.02 and transfer_r2 > 0.15:
                    verdict = "independent residual candidate"
                else:
                    verdict = "weak residual structure"

                results.append({
                    "system": system_name,
                    "task": task_name,
                    "k": k,
                    "eval": eval_label,
                    "sample_used": m if eval_label != "mixed" else 2 * m,
                    "r2_basic": r2_basic,
                    "r2_ctx": r2_ctx,
                    "corr_uncertainty": corr_unc,
                    "corr_margin_proxy": corr_margin,
                    "corr_error": corr_err,
                    "transfer_r2": transfer_r2,
                    "orth_best_alpha": best_alpha,
                    "orth_gain": orth_gain,
                    "residual_only_overlap": resid_only["overlap"],
                    "residual_only_accuracy": resid_only["accuracy"],
                    "verdict": verdict,
                })

len(results)

## Quick diagnostic

In [ ]:
sorted(set(r["system"] for r in results)), sorted(set(r["task"] for r in results)), sorted(set(r["k"] for r in results)), sorted(set(r["eval"] for r in results))

## Plot 1 · Residual predictability \(R^2\) across scale

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)

for ax, system_name in zip(axes, ["entropy", "unevenness"]):
    rows = [r for r in results if r["system"] == system_name and r["task"] == "zeta_vs_GUE" and r["eval"] == "mixed"]
    rows = sorted(rows, key=lambda x: x["k"])
    ax.plot([r["k"] for r in rows], [r["r2_basic"] for r in rows], marker="o", label="basic")
    ax.plot([r["k"] for r in rows], [r["r2_ctx"] for r in rows], marker="o", label="contextual")
    ax.set_title(system_name)
    ax.set_xlabel("window size k")
    ax.set_ylabel("residual predictability $R^2$")
    ax.legend()

plt.tight_layout()
plt.show()

## Plot 2 · Residual vs core uncertainty correlations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)

for ax, system_name in zip(axes, ["entropy", "unevenness"]):
    rows = [r for r in results if r["system"] == system_name and r["task"] == "zeta_vs_GUE" and r["eval"] == "mixed"]
    rows = sorted(rows, key=lambda x: x["k"])
    ax.plot([r["k"] for r in rows], [r["corr_uncertainty"] for r in rows], marker="o", label="corr with uncertainty")
    ax.plot([r["k"] for r in rows], [r["corr_error"] for r in rows], marker="o", label="corr with error")
    ax.plot([r["k"] for r in rows], [r["corr_margin_proxy"] for r in rows], marker="o", label="corr with inverse margin")
    ax.set_title(system_name)
    ax.set_xlabel("window size k")
    ax.set_ylabel("correlation")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Plot 3 · Cross-condition transfer heatmaps

In [ ]:
def build_transfer_matrix(system_name, task_name):
    rows = [r for r in results if r["system"] == system_name and r["task"] == task_name]
    return np.array([
        [next(r for r in rows if r["k"] == k and r["eval"] == "low")["transfer_r2"] for k in window_sizes],
        [next(r for r in rows if r["k"] == k and r["eval"] == "high")["transfer_r2"] for k in window_sizes],
        [next(r for r in rows if r["k"] == k and r["eval"] == "mixed")["transfer_r2"] for k in window_sizes],
    ])

fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharex=True, sharey=True)

for i, system_name in enumerate(["entropy", "unevenness"]):
    for j, task_name in enumerate(["zeta_vs_GUE", "zeta_vs_Poisson"]):
        ax = axes[i, j]
        M = build_transfer_matrix(system_name, task_name)
        im = ax.imshow(M, aspect="auto", origin="upper")
        ax.set_title(f"{system_name} · {task_name}")
        ax.set_xticks(np.arange(len(window_sizes)), window_sizes)
        ax.set_yticks(np.arange(3), ["low eval", "high eval", "mixed eval"])

fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85, label="cross-condition transfer $R^2$")
plt.tight_layout()
plt.show()

## Plot 4 · Orthogonal residual gain over core

In [ ]:
def build_orth_gain_matrix(system_name, task_name):
    rows = [r for r in results if r["system"] == system_name and r["task"] == task_name]
    return np.array([
        [next(r for r in rows if r["k"] == k and r["eval"] == "low")["orth_gain"] for k in window_sizes],
        [next(r for r in rows if r["k"] == k and r["eval"] == "high")["orth_gain"] for k in window_sizes],
        [next(r for r in rows if r["k"] == k and r["eval"] == "mixed")["orth_gain"] for k in window_sizes],
    ])

fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharex=True, sharey=True)

for i, system_name in enumerate(["entropy", "unevenness"]):
    for j, task_name in enumerate(["zeta_vs_GUE", "zeta_vs_Poisson"]):
        ax = axes[i, j]
        M = build_orth_gain_matrix(system_name, task_name)
        im = ax.imshow(M, aspect="auto", origin="upper")
        ax.set_title(f"{system_name} · {task_name}")
        ax.set_xticks(np.arange(len(window_sizes)), window_sizes)
        ax.set_yticks(np.arange(3), ["low eval", "high eval", "mixed eval"])

fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85, label="best orthogonal residual gain")
plt.tight_layout()
plt.show()

## Plot 5 · Residual-only classifier comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)

for ax, task_name in zip(axes, ["zeta_vs_GUE", "zeta_vs_Poisson"]):
    rows = [r for r in results if r["task"] == task_name and r["eval"] == "mixed"]
    rows = sorted(rows, key=lambda x: (x["system"], x["k"]))
    labels = [f'{r["system"][:3]}-{r["k"]}' for r in rows]
    vals = [r["residual_only_overlap"] for r in rows]
    ax.bar(np.arange(len(vals)), vals)
    ax.set_title(task_name)
    ax.set_ylabel("residual-only overlap")
    ax.set_xticks(np.arange(len(vals)), labels, rotation=45)

plt.tight_layout()
plt.show()

## Completeness summary table

In [ ]:
summary_rows = []
for system_name in ["entropy", "unevenness"]:
    for task_name in ["zeta_vs_GUE", "zeta_vs_Poisson"]:
        for k in window_sizes:
            row = next(r for r in results if r["system"] == system_name and r["task"] == task_name and r["eval"] == "mixed" and r["k"] == k)
            summary_rows.append({
                "system": system_name,
                "task": task_name,
                "k": k,
                "r2_ctx": row["r2_ctx"],
                "corr_error": row["corr_error"],
                "transfer_r2": row["transfer_r2"],
                "orth_gain": row["orth_gain"],
                "residual_only_overlap": row["residual_only_overlap"],
                "verdict": row["verdict"],
            })
summary_rows

## Compact summary

In [ ]:
summary = {
    "window_sizes": window_sizes,
    "feature_family": feature_family,
    "sample_size_target": sample_size,
    "height_block": f"{height_block[0]+1}-{height_block[1]}",
    "alpha_grid": list(map(float, alpha_grid)),
    "systems": list(systems.keys()),
    "tasks": list(tasks.keys()),
    "summary_rows": summary_rows,
}
summary

## Reading guide

- **High residual predictability \(R^2\)** means the residual is already largely explained by the current feature space.
- **Strong correlation with uncertainty/error** means the residual may just track where the core is unsure.
- **Weak transfer** means the residual is condition-local rather than reusable.
- **Near-zero orthogonal gain** means the unexplained part of the residual adds little or nothing to the decision boundary.
- **Weak residual-only performance** means the residual does not define an independent classifier.

## Completeness rule used here

Residual is labeled:
- **redundant** if predictability is high, orthogonal gain is near zero, and residual-only overlap is weak
- **independent residual candidate** if orthogonal gain and transfer are both positive
- **weak residual structure** otherwise

## Next directions

A natural next notebook could do one of these:

1. expand the feature space with genuinely new observables
2. test nonlinear residual predictability
3. compare completeness across feature families
4. map orthogonal residuals directly in feature space